# SQL Testing Patterns

## Overview
This notebook covers the full lifecycle of SQL testing: test-driven development (write tests first), regression testing (catch changes to existing queries), and CI/CD integration (automated testing in pipelines).

**Testing maturity levels:**

| Level | What you have |
|---|---|
| 0 — None | No tests; data quality issues discovered by users |
| 1 — Ad hoc | Manual spot checks; not repeatable |
| 2 — Structural | unique, not_null, accepted_values on all models |
| 3 — Business rules | Custom tests for domain logic (premiums, dates, ranges) |
| 4 — Automated CI | Tests run on every PR; failures block deployment |
| 5 — Monitored | DQ trends tracked over time; alerting on regressions |

---

In [ ]:
import sqlite3, json
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Mixed dataset: insurance + healthcare + financial transactions
CREATE TABLE customers (
    customer_id  TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    email        TEXT,
    province     TEXT NOT NULL,
    segment      TEXT NOT NULL,   -- 'retail','commercial','group'
    created_at   TEXT NOT NULL
);

CREATE TABLE policies (
    policy_id    TEXT PRIMARY KEY,
    customer_id  TEXT NOT NULL REFERENCES customers(customer_id),
    product_line TEXT NOT NULL,   -- 'auto','home','life','health'
    premium      REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'active','lapsed','cancelled'
    start_date   TEXT NOT NULL,
    end_date     TEXT
);

CREATE TABLE claims (
    claim_id     TEXT PRIMARY KEY,
    policy_id    TEXT NOT NULL REFERENCES policies(policy_id),
    claim_date   TEXT NOT NULL,
    amount       REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'open','approved','denied','paid'
    approved_by  TEXT
);

CREATE TABLE lab_results (
    result_id    TEXT PRIMARY KEY,
    patient_id   TEXT NOT NULL,
    test_name    TEXT NOT NULL,
    result_value REAL,
    unit         TEXT,
    result_date  TEXT NOT NULL,
    flagged      INTEGER DEFAULT 0
);

-- Intentionally seeded with some data quality issues for demos
""")

# Clean customers
customers = [
    ('C001','Alice Nguyen',    'alice@email.com',  'ON','retail',    '2022-01-15'),
    ('C002','Bob Harrington',  'bob@email.com',    'BC','commercial','2022-03-01'),
    ('C003','Fatima Al-Rashid','fatima@email.com', 'QC','group',     '2022-06-10'),
    ('C004','James MacLeod',   'james@email.com',  'NS','retail',    '2023-01-20'),
    ('C005','Mei-Ling Chen',   'mei@email.com',    'AB','commercial','2023-04-05'),
    ('C006','David Park',      None,               'ON','retail',    '2024-01-01'),  # null email
    ('C007','Sandra Okafor',   'sandra@email.com', 'ON','retail',    '2024-02-15'),
]
conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", customers)

# Policies — including some with data issues
policies = [
    ('POL-001','C001','auto',  1200.0,'active',   '2023-01-15','2024-01-15'),
    ('POL-002','C001','home',  1800.0,'active',   '2023-03-01','2024-03-01'),
    ('POL-003','C002','auto',  2400.0,'active',   '2022-06-01','2023-06-01'),
    ('POL-004','C003','life',   600.0,'lapsed',   '2022-09-01','2023-09-01'),
    ('POL-005','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),
    ('POL-006','C005','health',4800.0,'active',   '2023-05-01','2024-05-01'),
    ('POL-007','C006','home',  2100.0,'cancelled','2024-01-15','2024-06-15'),
    ('POL-008','C007','auto',  1100.0,'active',   '2024-03-01','2025-03-01'),
    ('POL-009','C002','home',  -500.0,'active',   '2024-01-01','2025-01-01'),  # negative premium (data issue)
    ('POL-010','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),  # duplicate of POL-005
]
conn.executemany("INSERT INTO policies VALUES (?,?,?,?,?,?,?)", policies)

# Claims
claims = [
    ('CLM-001','POL-001','2023-06-15',2200.0,'paid',    'Dr. Lee'),
    ('CLM-002','POL-003','2023-08-01', 500.0,'approved','Dr. Pham'),
    ('CLM-003','POL-005','2023-11-20',8500.0,'paid',    'Dr. Singh'),
    ('CLM-004','POL-006','2024-01-10', 350.0,'open',    None),
    ('CLM-005','POL-001','2024-02-28',1100.0,'denied',  'Dr. Lee'),
    ('CLM-006','POL-008','2024-04-15', 750.0,'approved','Dr. Pham'),
    ('CLM-007','POL-003','2024-05-01',99999.0,'open',   None),  # suspiciously large
]
conn.executemany("INSERT INTO claims VALUES (?,?,?,?,?,?)", claims)

# Lab results — with some quality issues
labs = [
    ('LAB-001','P001','HbA1c',    7.2,'%',        '2023-10-01',1),
    ('LAB-002','P001','eGFR',    68.0,'mL/min',   '2023-10-01',0),
    ('LAB-003','P002','HbA1c',    8.4,'%',        '2024-01-10',1),
    ('LAB-004','P002','Creatinine',115.0,'umol/L','2024-01-10',1),
    ('LAB-005','P003','HbA1c',    None,'%',        '2024-02-15',0),  # null value
    ('LAB-006','P001','HbA1c',    7.2,'%',        '2023-10-01',1),  # duplicate of LAB-001
    ('LAB-007','P004','eGFR',   -5.0,'mL/min',   '2024-03-01',0),  # impossible negative
    ('LAB-008','P003','HbA1c',    9.1,'%',        '2024-02-15',1),
]
conn.executemany("INSERT INTO lab_results VALUES (?,?,?,?,?,?,?)", labs)
conn.commit()

print("Testing dataset loaded:")
for tbl in ['customers','policies','claims','lab_results']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<15s}: {n} rows")
print()
print("Known data quality issues seeded:")
issues = [
    ("customers",    "C006 has NULL email"),
    ("policies",     "POL-009 has negative premium (-500)"),
    ("policies",     "POL-010 is a near-duplicate of POL-005"),
    ("claims",       "CLM-007 has suspiciously large amount (99999)"),
    ("lab_results",  "LAB-005 has NULL result_value"),
    ("lab_results",  "LAB-006 is an exact duplicate of LAB-001"),
    ("lab_results",  "LAB-007 has impossible negative eGFR (-5)"),
]
for tbl, issue in issues:
    print(f"  [{tbl}] {issue}")


---
## Test-driven SQL development

In [ ]:
print("=== Test-driven SQL development ===")
print()

print("TDD workflow for SQL:")
steps = [
    ("1. Write the test first",
     "Define what 'correct' looks like before writing the query",
     "SELECT COUNT(*) FROM monthly_loss_ratio WHERE loss_ratio < 0  -- should return 0"),
    ("2. Run — expect failure",
     "Test should fail because the model doesn't exist yet",
     "Relation 'monthly_loss_ratio' does not exist"),
    ("3. Write minimal SQL to pass",
     "Build the model; run tests; iterate",
     "CREATE VIEW monthly_loss_ratio AS SELECT ... FROM fact_policy JOIN ..."),
    ("4. Run — expect pass",
     "All assertions green",
     "0 rows returned from every test query"),
    ("5. Refactor with confidence",
     "Optimise the query; tests catch regressions",
     "Replace subquery with CTE; add partition; tests still pass"),
]
for step, desc, example in steps:
    print(f"  {step}")
    print(f"    {desc}")
    print(f"    e.g. {example}")
    print()

print("Minimal test harness in Python (no dbt required):")
harness_code = '''
import sqlite3

def run_test_suite(conn, tests):
    """
    tests: list of (name, sql) tuples
    SQL should return rows only on FAILURE.
    """
    results = {'passed': [], 'failed': []}
    for name, sql in tests:
        rows = conn.execute(sql).fetchall()
        if rows:
            results['failed'].append((name, len(rows), rows[:3]))
        else:
            results['passed'].append(name)

    print(f"Results: {len(results['passed'])} passed, {len(results['failed'])} failed")
    for name in results['passed']:
        print(f"  PASS  {name}")
    for name, n, sample in results['failed']:
        print(f"  FAIL  {name}  ({n} violations)")
        for row in sample:
            print(f"        {dict(row)}")

    return len(results['failed']) == 0

POLICY_TESTS = [
    ("no_negative_premiums",
     "SELECT policy_id, premium FROM policies WHERE premium < 0"),
    ("valid_status",
     "SELECT policy_id, status FROM policies WHERE status NOT IN ('active','lapsed','cancelled')"),
    ("no_null_customer_id",
     "SELECT policy_id FROM policies WHERE customer_id IS NULL"),
    ("referential_integrity",
     """SELECT p.policy_id FROM policies p
        LEFT JOIN customers c ON c.customer_id = p.customer_id
        WHERE c.customer_id IS NULL"""),
]

all_passed = run_test_suite(conn, POLICY_TESTS)
import sys; sys.exit(0 if all_passed else 1)   # non-zero exit for CI pipelines
'''
print(harness_code)


---
## Regression testing

In [ ]:
print("=== Regression testing: catch query changes ===")
print()

print("Regression test pattern: snapshot expected output, compare on each run")
print()

# Step 1: Capture a baseline
baseline = conn.execute("""
    SELECT province,
           COUNT(DISTINCT customer_id) AS n_customers,
           COUNT(p.policy_id)          AS n_policies,
           ROUND(SUM(p.premium), 2)    AS total_premium
    FROM   customers c
    LEFT JOIN policies p ON p.customer_id = c.customer_id
    GROUP  BY province
    ORDER  BY province
""").fetchall()

print("Baseline snapshot (captured 2024-03-01):")
print(f"  {'province':<10s}  {'n_customers':>12s}  {'n_policies':>11s}  {'total_premium':>14s}")
print("  " + "-"*50)
for r in baseline:
    print(f"  {r['province']:<10s}  {r['n_customers']:>12d}  {r['n_policies']:>11d}  ${r['total_premium']:>13,.2f}")

print()
print("Regression test: compare current output to baseline")

import json
baseline_dict = {r['province']: dict(r) for r in baseline}

# Simulate a regression (re-running the same query should match)
current = conn.execute("""
    SELECT province,
           COUNT(DISTINCT customer_id) AS n_customers,
           COUNT(p.policy_id)          AS n_policies,
           ROUND(SUM(p.premium), 2)    AS total_premium
    FROM   customers c
    LEFT JOIN policies p ON p.customer_id = c.customer_id
    GROUP  BY province
    ORDER  BY province
""").fetchall()

all_match = True
for r in current:
    prov = r['province']
    if prov in baseline_dict:
        b = baseline_dict[prov]
        if dict(r) != b:
            print(f"  REGRESSION in {prov}: {dict(r)} != {b}")
            all_match = False

if all_match:
    print("  PASS ✓ All province rows match baseline")

print()
print("Storing baselines in PostgreSQL:")
print("""
  CREATE TABLE test_baselines (
      test_name    TEXT NOT NULL,
      captured_at  TIMESTAMPTZ NOT NULL DEFAULT NOW(),
      expected_json JSONB NOT NULL,
      PRIMARY KEY (test_name)
  );

  -- Capture baseline:
  INSERT INTO test_baselines (test_name, expected_json)
  SELECT 'province_summary',
         jsonb_agg(row_to_json(sub.*))
  FROM (
      SELECT province, COUNT(*) AS n_customers
      FROM   customers
      GROUP  BY province ORDER BY province
  ) sub
  ON CONFLICT (test_name) DO UPDATE
      SET expected_json = EXCLUDED.expected_json,
          captured_at   = NOW();
""")


---
## CI/CD integration

In [ ]:
print("=== CI/CD integration patterns ===")
print()

print("dbt in a CI pipeline (GitHub Actions):")
gh_actions = """
# .github/workflows/dbt_ci.yml
name: dbt CI

on:
  pull_request:
    branches: [main]

jobs:
  dbt-test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.11'

      - name: Install dbt
        run: pip install dbt-postgres

      - name: Run dbt build (models + tests)
        env:
          DBT_HOST: ${{ secrets.DB_HOST }}
          DBT_USER: ${{ secrets.DB_USER }}
          DBT_PASS: ${{ secrets.DB_PASS }}
        run: |
          dbt deps
          dbt build --target ci --fail-fast

      - name: Upload test results
        if: failure()
        uses: actions/upload-artifact@v3
        with:
          name: dbt-test-results
          path: target/run_results.json
"""
print(gh_actions)

print("Custom SQL test runner for CI (no dbt):")
ci_runner = """
# test_runner.py -- run from CI pipeline
import sqlite3, sys, json, datetime

def run_tests(db_path, test_file):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    tests  = json.load(open(test_file))
    passed, failed = [], []

    for test in tests:
        rows = conn.execute(test['sql']).fetchall()
        if rows:
            failed.append({'name': test['name'], 'violations': len(rows)})
        else:
            passed.append(test['name'])

    report = {
        'timestamp': datetime.datetime.now().isoformat(),
        'passed':    len(passed),
        'failed':    len(failed),
        'failures':  failed
    }
    print(json.dumps(report, indent=2))
    sys.exit(0 if not failed else 1)   # non-zero = CI failure

# tests.json format:
# [{"name": "no_negative_premiums",
#   "sql":  "SELECT policy_id FROM policies WHERE premium < 0"}]
"""
print(ci_runner)

print("Key CI testing principles:")
principles = [
    ("Run tests on every PR",         "Catch regressions before merging to main"),
    ("Fail fast (--fail-fast)",       "Stop on first failure; surface issues quickly"),
    ("Tag critical tests",            "dbt tag:critical — run on every commit; full suite nightly"),
    ("Store failure rows",            "dbt --store-failures: failing records land in DWH for review"),
    ("Non-zero exit on failure",      "CI pipeline must fail if tests fail; not just print warnings"),
    ("Test on a CI-specific schema",  "Don't run destructive tests against production"),
    ("Version control your tests",    "Tests live in Git alongside models — always in sync"),
]
for principle, desc in principles:
    print(f"  {principle:<32s}  {desc}")


---
## Common Pitfalls

**1. Writing tests after the model is already in production**
Retrofitting tests onto an existing model that is already trusted by the business frequently reveals pre-existing data quality issues that are politically difficult to fix. Write tests before or alongside the model — not as an afterthought.

**2. Tests that always pass regardless of data**
A test harness that never reports failures provides false confidence. Periodically inject a known bad row into a test dataset and verify that the test catches it. Untested tests are not tests.

**3. CI pipeline that warns on test failure instead of failing**
A CI pipeline that reports test failures as warnings but still deploys the code trains developers to ignore the warnings. Test failures must fail the build and block the merge. Use `dbt build --fail-fast` or `sys.exit(1)` in custom runners.

**4. Regression baselines that are never updated**
Regression tests compare output to a captured baseline. If business logic legitimately changes, the baseline must be updated — or tests will permanently fail for the wrong reason. Version-control baselines and update them deliberately with a code review, not silently.

**5. Running all tests on every commit (slow CI)**
A full test suite on a 100-table DWH can take 30+ minutes. Use tags to split tests: run `tag:critical` (fast, high-impact) on every commit; run the full suite nightly or on main-branch merges only.

**6. No ownership assigned to failing tests**
When a test fails in CI, it must be clear who is responsible for fixing it. Add an `owner` field to each test in your test registry and route failures to the right team via Slack or PagerDuty. Unowned failures get ignored.


---
*sql_methods_library - Samantha McGarrigle*